# Advanced Analytics

## Mutual Fund Performance Analytics

Prepared by: Besi Karthik

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Load Datasets

In [3]:
nav = pd.read_csv(RAW_FOLDER / "02_nav_history.csv")

nav["date"] = pd.to_datetime(nav["date"])
nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

print(nav.head())

      amfi_code       date       nav  daily_return
5750     100016 2022-01-03  520.4608           NaN
5751     100016 2022-01-04  515.0971     -0.010306
5752     100016 2022-01-05  521.7239      0.012865
5753     100016 2022-01-06  515.7880     -0.011377
5754     100016 2022-01-07  515.1639     -0.001210


## Historical Value at Risk (VaR)& (CVar)

In [4]:

print("\nTotal Schemes:")
print(nav["amfi_code"].nunique())
print("\n===== HISTORICAL VaR & CVaR =====")

var_results = []

for code, fund in nav.groupby("amfi_code"):

    returns = fund["daily_return"].dropna()

    if len(returns) == 0:
        continue

    var_95 = np.percentile(returns, 5)

    cvar_95 = returns[returns <= var_95].mean()

    var_results.append({
        "amfi_code": code,
        "VaR_95 (%)": round(var_95 * 100, 2),
        "CVaR_95 (%)": round(cvar_95 * 100, 2)
    })

var_df = pd.DataFrame(var_results)

print(var_df)

var_df.to_csv("var_cvar_report.csv", index=False)

print("\nVaR & CVaR report saved successfully!")


Total Schemes:
40

===== HISTORICAL VaR & CVaR =====
    amfi_code  VaR_95 (%)  CVaR_95 (%)
0      100016       -1.44        -1.81
1      100025       -0.38        -0.50
2      100033       -1.90        -2.35
3      101206       -1.33        -1.74
4      101207       -2.60        -3.25
5      101208       -0.03        -0.04
6      102885       -1.26        -1.55
7      102886       -1.92        -2.33
8      102887       -1.52        -1.94
9      118632       -1.40        -1.76
10     118633       -1.34        -1.66
11     118634       -2.54        -3.23
12     118635       -1.26        -1.62
13     118636       -0.38        -0.49
14     119092       -1.37        -1.73
15     119093       -1.42        -1.75
16     119094       -1.85        -2.43
17     119095       -2.62        -3.17
18     119120       -0.39        -0.50
19     119551       -1.28        -1.64
20     119552       -1.35        -1.73
21     119598       -2.45        -3.06
22     119599       -2.69        -3.24
23     120

## Rolling Sharpe Ratio

In [7]:
print("\n===== ROLLING 90-DAY SHARPE RATIO =====")

plt.figure(figsize=(14,7))

top5_funds = nav["amfi_code"].unique()[:5]

for code in top5_funds:

    fund = nav[nav["amfi_code"] == code].copy()

    rolling_mean = fund["daily_return"].rolling(90).mean()

    rolling_std = fund["daily_return"].rolling(90).std()

    fund["rolling_sharpe"] = (
        rolling_mean / rolling_std
    ) * np.sqrt(252)

    plt.plot(
        fund["date"],
        fund["rolling_sharpe"],
        label=str(code)
    )

plt.title("Rolling 90-Day Sharpe Ratio (Top 5 Funds)")
plt.xlabel("Date")
plt.ylabel("Sharpe Ratio")
plt.legend(title="AMFI Code")
plt.grid(True)

plt.tight_layout()

plt.savefig("rolling_sharpe_chart.png")

plt.close()

print("\nRolling Sharpe chart saved successfully!")



===== ROLLING 90-DAY SHARPE RATIO =====

Rolling Sharpe chart saved successfully!


## Investor Cohort Analysis

In [8]:

transactions = pd.read_csv(
    RAW_FOLDER / "08_investor_transactions.csv"
)

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

print("\nFIRST 5 ROWS:")
print(transactions.head())

print("\nCOLUMNS:")
print(transactions.columns.tolist())
print("\n===== INVESTOR COHORT ANALYSIS =====")

transactions["cohort_year"] = (
    transactions.groupby("investor_id")["transaction_date"]
    .transform("min")
    .dt.year
)

sip_transactions = transactions[
    transactions["transaction_type"] == "SIP"
]

avg_sip = (
    sip_transactions
    .groupby("cohort_year")["amount_inr"]
    .mean()
    .reset_index(name="Average SIP Amount")
)

total_invested = (
    transactions
    .groupby("cohort_year")["amount_inr"]
    .sum()
    .reset_index(name="Total Invested")
)

top_fund = (
    transactions
    .groupby(["cohort_year", "amfi_code"])
    .size()
    .reset_index(name="Count")
)

top_fund = (
    top_fund
    .sort_values(["cohort_year", "Count"], ascending=[True, False])
    .groupby("cohort_year")
    .first()
    .reset_index()
)

cohort_report = (
    avg_sip
    .merge(total_invested, on="cohort_year")
    .merge(
        top_fund[["cohort_year", "amfi_code"]],
        on="cohort_year"
    )
)

print(cohort_report)

cohort_report.to_csv(
    "investor_cohort_analysis.csv",
    index=False
)


FIRST 5 ROWS:
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Veri

## SIP Continuity Analysis

In [9]:
sip = transactions[
    transactions["transaction_type"] == "SIP"
].copy()

sip = sip.sort_values(
    ["investor_id", "transaction_date"]
)

sip["gap_days"] = (
    sip.groupby("investor_id")["transaction_date"]
    .diff()
    .dt.days
)

sip_summary = (
    sip.groupby("investor_id")
    .agg(
        sip_transactions=("transaction_date", "count"),
        average_gap_days=("gap_days", "mean")
    )
    .reset_index()
)

sip_summary = sip_summary[
    sip_summary["sip_transactions"] >= 6
]

sip_summary["status"] = np.where(
    sip_summary["average_gap_days"] > 35,
    "At Risk",
    "Active"
)

print(sip_summary)

sip_summary.to_csv(
    "sip_continuity_report.csv",
    index=False
)

     investor_id  sip_transactions  average_gap_days   status
3      INV000004                 6         85.400000  At Risk
7      INV000008                 6         70.400000  At Risk
9      INV000010                 6         64.800000  At Risk
10     INV000011                 7         40.166667  At Risk
11     INV000012                 8         57.000000  At Risk
...          ...               ...               ...      ...
4746   INV004984                 7         75.333333  At Risk
4748   INV004986                 7         81.333333  At Risk
4754   INV004992                 7         81.500000  At Risk
4758   INV004996                10         46.333333  At Risk
4759   INV004997                 7         71.000000  At Risk

[1362 rows x 4 columns]


## Sector Diversification (HHI)

In [10]:

holdings = pd.read_csv(
    RAW_FOLDER / "09_portfolio_holdings.csv"
)

print("\nPortfolio Holdings Columns:")
print(holdings.columns.tolist())

print("\nFirst 5 Rows:")
print(holdings.head())


Portfolio Holdings Columns:
['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct', 'market_value_cr', 'current_price_inr', 'portfolio_date']

First 5 Rows:
   amfi_code stock_symbol                stock_name       sector  weight_pct  \
0     119551    POWERGRID    Power Grid Corporation    Utilities       13.85   
1     119551     HDFCBANK             HDFC Bank Ltd      Banking       11.19   
2     119551       GRASIM     Grasim Industries Ltd  Diversified        9.90   
3     119551      DRREDDY  Dr. Reddy's Laboratories       Pharma        4.76   
4     119551   ASIANPAINT          Asian Paints Ltd       Paints       10.25   

   market_value_cr  current_price_inr portfolio_date  
0           737.09            6011.08     2025-12-31  
1            88.97            1074.65     2025-12-31  
2           208.45            5964.59     2025-12-31  
3           161.32            3748.82     2025-12-31  
4           725.90            1321.45     2025-12-31  


## Fund Recommendation

In [12]:

fund_master = pd.read_csv(
    RAW_FOLDER / "01_fund_master.csv"
)

print("\nFund Master Columns:")
print(fund_master.columns.tolist())

print("\nRisk Categories:")
print(fund_master["risk_category"].unique())
sharpe_df = pd.read_csv("sharpe_ratio.csv")


Fund Master Columns:
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

Risk Categories:
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']
